# Практика · TF-IDF

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) ·
> Тест: [quiz.html](quiz.html)

Тут ми рахуємо **всі** числа, які називає лекція. Порядок такий самий, як у ній.

Що зробимо:

1. Зберемо корпус українських перекладів інтерфейсів і подивимось на нього очима.
2. Порахуємо IDF і знайдемо слова з найменшою й найбільшою вагою.
3. Перевіримо головне твердження теми: чи змінює IDF те, яке слово в документі головне.
4. Напишемо TF-IDF **своїми руками** і звіримо із `sklearn` до чотирнадцятого знака.
5. Заміряємо, що буває без логарифма, без згладжування і без нормалізації.
6. Зламаємо TF-IDF трьома різними способами — і кожен замір покажемо числом.
7. Зберемо підсумок усього блоку 1 в одну таблицю.

> ⏱ Заміряно: близько **80 секунд** на чотирьох ядрах без відеокарти, зошит
> запущено наодинці. Найдовше йде тричі повторений пошук по 20 000 документів
> і три прогони символьних триграм.

## 1 · Середовище

Перша клітинка друкує версії. Якщо в тебе інші — числа можуть трохи поїхати,
і краще знати про це одразу, а не наприкінці.

In [ ]:
import sys, re, math, glob, gettext, time, collections
import numpy as np
import scipy.sparse as sp
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("scipy   ", __import__("scipy").__version__)
print("sklearn ", sklearn.__version__)

## 2 · Корпус: українські переклади інтерфейсів

Кожна програма в Linux має файл перекладу `.mo`: там лежать пари «англійський
оригінал → український переклад». Ми беремо **переклади** як документи.

Чому саме цей корпус: це справжня українська мова, він лежить на диску (жодної
мережі) і не міняється між запусками. Чому він **не** універсальний: це вузький
домен — технічна лексика, короткі речення, багато наказового способу. Висновків
про мову взагалі з нього робити не можна, і далі ми побачимо, як саме це вилазить.

Якщо української локалі на машині немає, вмикається вбудований запасний корпус —
маленький, але достатній, щоб увесь зошит виконався до кінця.

In [ ]:
def load_system_corpus():
    """Читаємо всі .mo-файли української локалі. Повертаємо трійки
    (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                       # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for src, dst in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(src, str) and isinstance(dst, str) \
               and len(dst) > 30 and 'Project-Id' not in dst:
                docs.append((program, src, dst))
    return docs

FALLBACK = [
"перекодувати метадані в це кодування",
"Перевищено час очікування на введення-виведення з гнізда",
"Буде проігноровано, зберігається для сумісності",
"Не вдалося отримати дані про ємність акумулятора",
"Додає маломасштабний растр, подібний до локальної зернистості",
"Налаштувати таблицю клавіш кани",
"Додати тло документа до кожного перетвореного шару.",
"Обмежити загальне використання чорнил на аркуш паперу",
"Вікно віджету, яке реалізовано",
"Мінімальне розширення дочірнього елемента",
"Вам слід включити принаймні один диск як ціль встановлення.",
"швейцарська німецька мова жестів",
"Виконано забагато спроб отримання пароля",
"Андаманські і Нікобарські острови",
"Чи слід підвкладкам заповнювати виділену ділянку",
"Увімкнути помилку подвійного подавання через товщину паперу",
"Однорідний за розміром вертикально",
"Некоректний формат прав доступу",
"Неочікуване завершення формального виразу",
"Більше старих транзакцій немає.",
"Некоректні параметри режиму знімання",
"Тритлумити яскравість екрана після періоду неактивності",
"Використовувати спекуляцію даних після перезавантаження.",
"вказана адреса основного сервера ключів є некоректною",
"не вдалося створити знімок вікна",
"Порогове значення порожніх сторінок для програмного відкидання",
"не перевіряти шляхи символічних посилань на файли",
"Збільшений час очікування лампи",
"не є коректною числовою специфікацією",
"пароль користувача у форматі звичайного тексту",
"не вдалося отримати повідомлення від батьківського процесу",
"Точне налаштування балансу білого",
"Ширина паперу, на якому слід надрукувати дані",
"показувати також ідентифікатори груп",
"УВАГА: цілісність повідомлення не захищено",
"Опале листя восени або щось вкрите живим листям",
"Під час приготування документа сталася помилка",
"помилка у додатку підтвердження",
"Відстань за вертикаллю між двома дочірніми віджетами",
"не можна комбінувати кілька визначників фільтра",
"Не вдалося оновити файл налаштування.",
"в заголовку файлу не знайдено магічного рядка",
"Динамічна область, найближчий предмет",
"несумісні типи в бінарному виразі",
"Не вдалося обробити відповідь сервера",
"Не вдалося побудувати назву інтерфейсу",
"Визначає, чи може бути встановлено часткову відповідність",
"вивести список усіх пакунків неактивних потоків модуля",
"не вказано модель захисту під час використання декількох міток",
"Показати список відомих областей",
"Не вдалося отримати список мовних розширень модуля",
"Не вдалося завантажити модуль обробки",
"Назва групи для перетягування вкладки",
"Неочікуваний передчасний кінець потоку",
"Датчик кольорової ділянки з двома чипами",
"Вивести відомості щодо користувача і перевірити розпізнавання",
"Ідентифікатор повідомлення у каталозі",
"Перемістити обчислення незмінних у петлях за межі петель.",
"Не вдалося виконати читання з монітора",
"Мінімальна ширина кнопок в контейнері",
"параметри містив пароль з порожньою назвою",
"Буде встановлено при наступному вході",
"Попереджати про невіртуальні деструктори.",
"Показати коротке повідомлення щодо користування для команди",
"Апаратний блок призначення не збігається із джерелом",
"Ізольована мережа, лише внутрішня маршрутизація",
"Висота відеозапису, одержаного з камери, в пікселях",
"Процесором основної системи не надаються потрібні можливості",
"Немає повідомлення бажаного типу",
"Чи інтеграцію з оболонкою ввімкнено",
"Показувати під час запуску стовпчик використання процесору.",
"Не можна вилучати загальносистемні розширення",
"Команда автоматичної компенсації спалаху",
"Чи показується розмір вибраного шрифту у позначці",
"Не вдалося створити зображення попереднього перегляду друку.",
"кореневий сертифікат було позначено як надійний",
"Перейти до попереднього елемента списку",
"Чи показувати кнопку закривання на панелі інструментів",
"Базовий рельєф з двома джерелами світла",
"недійсні типи у конвертації до цілого числа",
"Не вдалося виконати пошук файла",
"У назві домену не повинно міститися символів розриву рядка",
"регістр джерела збігається з основою зворотного запису",
"не вдалося завантажити дані символів",
"Експортувати обрізаний вміст у сторінці",
"читання користувацьких методів доступу",
"Масштабувати за центром сторінки",
"перевизначити типове розташування кореневої теки системи",
"обʼєкт призначення виділяється тут",
"Визначає відстань до точки призначення.",
"Під час спроби вивантаження підшляху метаданих:",
"Для встановлення усіх наданих випусків потрібен пристрій",
"вивести дані у зручному для читання форматі",
"Не вдалося отримати адреси фрагмента.",
"елемент масиву позначок не є рядком",
"Внутрішня помилка: невідома помилка",
"Автономний регіон Азорські острови",
"Встановити базовою одиницею виміру сантиметр",
"Рядок, що показано в позначці вкладки дочірнього елемента",
"Визначає спосіб малювання тіні навколо порту перегляду",
"помилка позиціонування голівки сканування",
"встановити ширину каналу перенесення після копіювання",
"Керування динамічними правами доступу",
"Друкувати виконані операції і код завершення команди",
"Неможливо започаткувати аналізатор файлів конфігурації.",
"Показати додаткові діагностичні дані",
"Прочитати кутку зі стандартного джерела вхідних даних",
"визначити батьківський запис поточного знімка",
"Не можна мати розділи, які перекриваються.",
"ігноруємо некоректний необроблений пароль",
"Неможливо перейменувати файли верхнього рівня",
"конфліктуючі рядки заміни для порожнього поля",
"Глянцевий фотопапір найвищої якості",
"мало бути вказано регістр цілих чисел",
"Клавіатурне скорочення для очищення підсвічування знайденого",
"Використовувати лише апаратні значення",
"В цьому файлі немає динамічного розділу.",
"Виберіть вузол для отримання форми входу",
"відповідного пристрою файлової системи не знайдено",
"Файл сертифіката для перевірки сервера",
"показати контрольні суми повідомлень",
"Не вдалося відкрити пристрій тимчасового сховища ключів.",
"Отримати криві з позначеного для заміни поточного гліфу",
"небезпечний непрямий виклик функції в межах атомної транзакції",
"використання застарілих параметрів налаштовування",
"і кожен тип може бути перетворений на інший",
"Символи запису музики знаменного співу",
"сміття після числового літерала",
"доступ до тимчасових індексів з інших сесій заблокований",
"Не вдалося прочитати директорію завантаження",
"Припустити, що переповнення вказівника обгортається.",
"Проведіть вашим пальцем вздовж пристрою для зчитування",
"Демонтувати пристрій, змонтований іншим користувачем",
"сертифікат не мав використовуватися для підписування",
"для зміни типу потрібно бути суперкористувачем",
"Визначає, чи слід вмикати пришвидшення обробки плоских полотен",
"Горно-Бадахшанська автономна область",
"Автоматичне злиття не спрацювало.",
"недостатньо даних експорту для читання",
"Завершення роботи за бажанням користувача",
"Створити проміжки у самоперетинах, як у кельтських вузлах",
"неможливо додати контракти до віртуальної функції",
"Для виконання дій з файлами слід пройти розпізнавання",
"сертифікат непридатний для підписування",
"Увімкнути евристику підрахунку залежностей в планувальнику.",
"Підтримки перемикання розділів у коді не передбачено.",
"Нова подія наступного понеділка",
"Взяти видимий колір і прозорість",
"Не вдається задовольнити всі обмеження на розділ.",
"подвійні константи не підтримуються",
"Накопичувати зсув для кожного стовпчика",
"Пересунути спосіб введення нижче",
"оновити, навіть якщо індекс містить не злиті записи",
"Назва файла залежностей, який слід створити",
"Перемістити плоский перегляд на кінець рядка",
"помилка під час виконання швидкого експорту",
"не вдалося отримати дані щодо типу долучення носія даних",
"не вдалося додати подію до черги обробки",
"Проведення ліворуч двома пальцями",
"Сертифікат не містить відкритого ключа",
"Перед використанням пристрій не було зарезервовано",
"елементи керування відтворенням та показом стану аудіо",
"типи не можуть бути визначені в умовах",
"некоректний заголовок прозорого підпису",
"Перелік символічних імен і еквівалентів кольорів.",
"Показати діаграму процесора як багатоярусну діаграму",
"Виберіть зірку, щоб залишити оцінку",
"Очікуємо на носій або джерело пакунків для встановлення",
"під час спроби визначити розмір файлової системи",
"Розмір кроку гучності для кожної зміни гучності",
"У автоматичному режимі явні зупинки ігноруються",
"Не вдалося обробити список сеансів",
"Стан апаратного захисту та відомості щодо нього",
"зовнішня команда для перевірки віртуальних оболонок",
"пропущено вираз для ширини паперу",
"Вимикання системи, коли інші користувачі ще у ній",
"Позначте, щоб зробити шрифт курсивним.",
"неприпустиме використання атрибутів у порожньому оголошенні",
"Довжина циклу блимання курсора, в мілісекундах",
"Завантажити систему у режимі єдиного користувача.",
"Надавачі даних щодо обмінних курсів",
"Операція встановлення позиції не підтримується для потоків",
"таку назву інтерфейсу зарезервовано",
"потрібні деякі комміти для відтворення",
"жодного псевдоніму для стовпця не було надано",
"Перетягніть, щоб змінити позицію у стосі ефектів контуру",
"Для вставлення розширення до вузла не вистачає місця",
"Іріан Джайя та Молуккські острови",
"Розпочато роботу з повторного трасування",
"Незбережені зміни буде втрачено без можливості відновлення.",
"Програмі не вдалося знайти жодного доступного пакунка.",
"Не вдалося встановити сталість інтерфейсу тунелю",
"Обмеження на вільний дисковий простір",
"Реєстраційні дані для доступу до реєстру джерела",
"Віджет піктограми для показу в пункті",
"не вдалося знайти звуковий модуль для звукового пристрою",
"Не вдалося перевірити кореневий хеш.",
"Одиниця вимірювання для відстані до цілі",
"Вибрати колір для варіантів з бази даних користувача",
"присвоєння виразу з типом масиву",
"показати дані щодо проблем із наданням залежностей",
"для параметра функції масиву необхідно вказати вираз довжини",
"Щоб відкрити пристрій, слід пройти розпізнавання",
"Скористатися файлом як вхідним списком дій",
"Не вдалося отримати кількість контрольних точок",
"Немає надзвичайної дії з монтування",
"Екран, на якому буде виведено це вікно",
"Попереджати, якщо простір адреси змінюється.",
"не запускати і не зупиняти служби",
"Не вдалося визначити час, коли цю дію було виконано востаннє",
"Порівняти версії, які вказано як аргументи.",
"НАЗВА РОЗТАШУВАННЯ - Додати віддалене сховище",
"засновано на вашому паролі для входу",
"Розмір значків у цій панелі інструментів",
"перевизначити метадані для поточного знімка",
"Ви ввели правильний пароль. Спробуйте знову.",
"Для завершення дії на диску недостатньо вільного місця",
"Виключити типові каталоги зі шляху пошуку файлів",
"Збирати інформацію про команди які виконуються.",
"Вивести діагностичну інформацію про оптимізацію.",
"Клацніть, щоб відшукати символ.",
"Додавання нового віртуального обладнання",
"Не забезпечено методів розпізнавання",
"Документація, яка може допомогти:",
"вказано базовий регістр, але нульовий",
"Пошук і заміна у межах документа",
"некоректний результат оптимізації фрагмента",
"Показувати розділи для обробки виключень",
"Вставлено новий рядок або стовпчик.",
"неможливо створити тимчасове відношення в не тимчасовій схемі",
"Вийти з оболонки і повернутися до головного меню",
"Неможливо скопіювати файл сам у себе.",
"Помилковий вираз поточного значення",
"Чи цей рядок заголовку слід сховати, коли вікно розгорнуто",
"Стиль підкреслення цього тексту",
"немає завдання із резервного копіювання домену",
"Помилка при виводі декодованого шаблону",
"Встановлює кількість часу для оновлення файлу журналу.",
"Не вдалося отримати дані щодо геометрії диска.",
"Розмір вектора не є цілочисельним кратним розміру компоненти",
"Не вдалося оновити мікропрограму:",
"Можливі значення параметра СТИЛЬ:",
"Інформація про властивості пристрою:",
"ціль не є вказівником або посиланням на клас",
"Визначити значення змінної за введеними користувачем даними.",
"вилучити пакунок або пакунки з вашої системи",
"Не вистачає властивості ідентифікатора ОС розгортання",
"Вказати процесор для моделі конвеєра.",
"файл з записами щодо даних користувачів",
"Ширина, в точках, ліній рівня вкладення та ліній сітки",
"Більше немає відвіданих посилань.",
"не є числом, використовуємо нуль.",
"Перегляд і налаштовування скорочень",
"Радіус вікна, яке буде проаналізовано",
"не дозволяється безіменне перелічування з областю видимості",
"Віджет, який зараз показано у стосі",
"Наступний вузол за порядком читання вузлів",
"Не записувати дані до бази даних журналу",
"потік перервано іншим потоком обробки",
"Несподівана бітова глибина для елементів мапи кольорів",
"Використовувати нетиповий шрифт для мовної панелі",
"імпортувати визначення таблиць зі стороннього серверу",
"обчислене значення не використовується",
"Вказано невідомий алгоритм або протокол.",
"Носій не має ідентифікатора, неможливо вилучити",
"не вдалося визначити символ екранування",
"Некоректний операнд: поточне значення використано як адресу.",
"кількість процесорів є надто великою",
"Початкова точка для визначення початкового кута",
"за допомогою зовнішнього скрипту компонування:",
"Не вибрано жодного джерела введення",
"Визначає, чи є видимим перемикач, який вмикає розгортання",
"Перейти до робочого простору праворуч",
"Модуль зараз виконує демонтування",
"Прилипання лише до вузла, найближчого до вказівника",
"Чи повинні розкривні елементи мати лінію відриву",
"Увімкнути або вимкнути екранну клавіатуру",
"не вказано архітектурного розширення",
"не вдалося створити вихідні файли",
"Ширина стовпчика сеансу процесу",
"для цього буфера сховища даних слід вказати місткість тому",
"Тип переспрямовування служби і мережі",
"Придатні до встановлення атрибути:",
"Успішно вимкнено запис віддаленого сховища",
"Колір переднього плану у вигляді рядка",
"Вказує які сповіщення показуються і що вони показують",
"Ви знайдете свій файл у каталозі Звантаження.",
"Не вдалося передати дані в вікно",
"назви можливостей, відокремлені комами",
"Створити чотири напрямні за краями поточної сторінки",
"не виводити список згорнутих ідентичних розділів",
"Не вдалося побудувати контекст обробки",
"Компілювати код для режиму великого порядку байтів.",
"Показати користувача, яким було внесено зміну",
"рядок починається або завершується на заборонений дефіс",
"отримати блокову статистику пристроїв для домену",
"Помилка введення-виведення під час розшифрування слоту ключів.",
"неможливо порівняти іменований канал з директорією",
"Немає відомостей щодо насильства у мультфільмах",
"Наразі ви редагуєте коміт при перебазуванні.",
"Можете залишити панельні вікна тут.",
"занадто багато значень у вказівці повернення",
"Будь ласка, введіть відомості щодо вашого вторинного ключа:",
"Вікно заблоковано. Натисніть, щоб унести зміни",
"Передчасний кінець регулярного виразу",
"Чи дозволяти зміну постачальника встановлених залежностей.",
"Помилка читання зображення з карти",
"Показати інформацію щодо вказаного файла",
"Підрозділів на основну кругову поділку:",
"не можна поєднувати пре- і постіндексування",
"Помилка під час спроби обробки даних вводу-виводу агента",
"Не вдалося прочитати вхідні дані користувача",
"глобальна кваліфікація імені класу недійсна",
"Скоригувати точку дотику дотичної",
"Довжина проміжку між стібками при показі стібків",
"неможливо підключити предка успадкування в якості секції",
"виконати команду у фоновому режимі",
"Комбінація клавіш для відкривання нової вкладки",
"невідома помилка запису в стандартний вивід",
"Чи слід показувати лінії рівня вкладення у віджеті",
"Щоб звантажити альбом, потрібно вказати адресу фонотеки.",
"Приведення дерева до початкового стану...",
"неприпустимий шлях до віддаленої служби",
"Рядок, який буде використано для некоректних символів",
"ваша поточна гілка виглядає пошкодженою",
"Позиція у символах, на якій слід показувати праве поле.",
"Закриті частини основного ключа зберігаються на картці.",
"Апаратна або програмна рухома крапка",
"Створити розділ на нерозподіленому просторі",
"додати більше причин тайм-ауту не можна",
"Датчик ділянки послідовності кольорів",
"Налаштовування мобільного широкосмугового пристрою",
"Помилка служби виявлення пристроїв.",
"Вимовляти координати комірок таблиці",
"Повернена помилка з порожнім тілом",
"не вдалося записати граф комітів",
"Низький заряд батареї гучномовця",
"мало бути використано попередньо індексований вираз",
"сертифікат має ПОМИЛКОВИЙ підпис",
"Додавання опорної точки градієнта",
"Розділювач полів - нульовий байт.",
"Загальний час завантаження у пристрій",
"доступ до тимчасових таблиць з інших сесій заблоковано",
"Не вдалося увійти до жодного з виявлених вузлів",
"Схоже, ви виміряли не ту смугу.",
"Помилка експортування растрових даних",
"не вдалося створити пару сокетів",
"Спроба виконання дії була невдалою",
"Перейти вниз на наступний рядок",
"Не вдалося створити ідентифікатор наступного класу",
"Генерувати інструкцію повернення в голій функції.",
"необхідний шаблон текстового пошуку",
"Не вдалося отримати список буферів",
"заборонене використання керівного регістра",
"Вихідний статус визначається ВИРАЗОМ.",
"Подробиці: проксі не було створено.",
"Пошук потрібних пакунків у сховищах",
"Номер акумулятора виходить за межі",
"некоректне значення кількості активних процесорів вузла",
"Стара версія програмного інтерфейсу",
"Пропустити поточне вікно під час пошуку",
"останній аргумент повинен бути негайним значенням",
"Вибирати тему кнопок зі стрілками у вікні варіантів",
"профіль петлі не може бути портом",
"виявлено помилкове кодування адреси",
"Немає придатного до використання жетона.",
"Подвоїти оптичну роздільну здатність",
"подальші попередження щодо багатобайтових символів придушено",
"перегляд неактивних та активних доменів",
"Показує версію сервера у вигляді цілого числа.",
"Перейти на рівень вгору ієрархією документа",
"суперечливі параметри визначення ширини",
"Показувати тимчасовий обрис контуру",
"непідтримуваний розмір змінної або значення заповнення",
"Отримати список всіх доступний профілів кольорів",
"Підтримка користування на малому екрані",
"Збереження документа як шаблону",
"Перейти до попереднього перехресного посилання",
"Не використовуйте апаратне з плаваючою комою.",
"регістр індексу перериває регістр перенесення",
"Неможливо створити директорію кешу метаданих.",
"Долучення до цієї області неможливе",
"Дата останнього доступу до файлу користувачем.",
"Запросити у користувача, якщо потрібна перевірка автентичності",
"не можна одночасно знищити і функцію і змінну",
"Є повний опис терміналу. Всі клавіші працюють.",
"Випадкова варіація довжини ліній побудови",
"зберігати коміти, які починаються порожніми",
"гігабайт,гігабайти,гігабайтів,ГБ",
"Наблизити ефект викликів функцій для спрощення аналізу.",
"Акумулятор не є сталим цілочисельним",
"Не вдалося експортувати перевизначення користувача",
"Перемкнутися на режим півширинних літер",
"другий, третій і четвертий аргументи повинні бути константами",
"показати список лише активних буферів",
"У потоці міститься недостатньо даних.",
"максимальний розмір кожного файла пакунка",
"Функція оптично-електронного перетворення",
"Мітка для пропозицій від засобу перевірки правопису",
"Показувати параметри вибору файла"
]

corpus = load_system_corpus()
if len(corpus) < 5000:
    # української локалі на машині немає — працюємо на вбудованому корпусі
    corpus = [('fallback', '', t) for t in FALLBACK]
    print("⚠️  системної локалі немає, працюємо на вбудованому корпусі")
else:
    print("шлях спрацював: /usr/share/locale/uk/LC_MESSAGES/*.mo")

programs = sorted(set(p for p, _, _ in corpus))
print("документів:", len(corpus))
print("програм:   ", len(programs))
print()
for program, source, target in corpus[:3]:
    print(f"[{program}] {source[:46]!r}\n         -> {target[:60]!r}")

## 3 · Що ми вважаємо словом

`CountVectorizer` і `TfidfVectorizer` за замовчуванням ріжуть текст правилом
`\b\w\w+\b`: послідовності з **двох і більше** «словесних» символів. Для української
це погано одразу з трьох причин. Воно викидає односимвольні прийменники «у», «з», «і» —
а це найчастіші слова мови. Воно лишає цифри й латиницю, тож буква `s` із підстановки
`%s` пролізає в словник і стає одним із найчастіших «слів» корпусу. І воно ріже слово
на апострофі: у наших файлах апостроф найчастіше записаний звичайним `'` (U+0027),
а це не «словесний» символ — тож «зʼєднання» розпадається на «з» плюс «єднання».

Тому весь блок 1 користується одним спільним правилом:

    TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"

Читається так: одна або більше українських літер, після яких **може** йти апостроф
і ще літери — і так скільки завгодно разів. Апостроф тут не символ слова, а звʼязка
**всередині** слова, тож «зʼєднання» лишається одним токеном. Усе інше — цифри,
латиниця, `%s`, розділові знаки — просто не збігається з шаблоном і зникає само.

Спільний шаблон важливий не через красу: теми блоку посилаються на числа одна одної,
і якби кожна різала текст по-своєму, ці числа перестали б сходитися.

In [ ]:
# спільний токенізатор усього блоку 1: українські літери,
# апостроф — звʼязка всередині слова, а не окремий символ
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
tokenize = re.compile(TOKEN_PATTERN).findall

documents = [target for _, _, target in corpus]

print("текст :", documents[17])
print("токени:", tokenize(documents[17].lower()))
print()
print("три місця, де стандартний шаблон sklearn поводиться інакше:")
DEFAULT = r"(?u)\b\w\w+\b"
for sample_text in ["з'єднання з сервером",
                    "у файлі знайдено 3 помилки",
                    "кодування UTF-8 не підтримується"]:
    print(f"  {sample_text!r}")
    print("     sklearn ->", re.findall(DEFAULT, sample_text.lower()))
    print("     наш     ->", tokenize(sample_text.lower()))

## 4 · Двадцять тисяч документів і три зерна

Повний корпус великий, а нам треба, щоб зошит виконувався за хвилину. Беремо
**20 000 випадкових документів** — і робимо це **тричі**, з зернами 0, 1 і 2.

Навіщо три зерна: далі ми будемо порівнювати способи зважування, і різниця між
ними має сенс тільки тоді, коли вона більша за розкид між зернами. Різниця, менша
за розкид, — це не різниця, а шум вибірки.

In [ ]:
SEEDS = (0, 1, 2)
N_DOCS = 20000

def sample_documents(seed, n=N_DOCS):
    """Ті самі 20 000 документів при тому самому зерні — на будь-якій машині."""
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(documents), min(n, len(documents)), replace=False)
    return [documents[i] for i in picked]

def count_matrix(texts):
    """Мішок слів із теми 04: рядок — документ, колонка — слово, клітинка — скільки разів."""
    vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
    counts = vectorizer.fit_transform(texts).tocsr()
    return vectorizer, counts

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    cells = counts.shape[0] * counts.shape[1]
    print(f"зерно {seed}: матриця {counts.shape[0]} x {counts.shape[1]}, "
          f"ненульових {counts.nnz}, заповнено {100*counts.nnz/cells:.4f} %")

Заповнено близько пʼяти сотих відсотка — та сама розрідженість, яку тема 04 назвала
причиною, а не оптимізацією. Зерно 1 дає рівно ті самі **14 446** ознак і **138 665**
ненульових, що й контрольний прогін теми 04: спільний токенізатор означає, що числа
блоку сходяться між темами.

Тримай це в голові: усе, що ми робимо далі, міняє **значення** в цих клітинках,
але не їхню кількість.

## 5 · IDF: хто важить мало, а хто багато

`TfidfVectorizer` кладе готові ваги в `idf_`. Формула `sklearn` така:

    idf(w) = ln((1 + N) / (1 + df(w))) + 1

де `N` — скільки всього документів, `df(w)` — у скількох із них слово трапилось
хоч раз. Одиниці й логарифм ми розберемо через два розділи; поки що просто
подивимось, кому скільки дісталося.

In [ ]:
def fit_tfidf(texts, **kwargs):
    vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN, **kwargs)
    matrix = vectorizer.fit_transform(texts).tocsr()
    return vectorizer, matrix

for seed in SEEDS:
    texts = sample_documents(seed)
    tfidf, _ = fit_tfidf(texts)
    words = tfidf.get_feature_names_out()
    idf = tfidf.idf_
    order = np.argsort(idf)
    lowest = ", ".join(f"{words[i]} {idf[i]:.2f}" for i in order[:5])
    at_top = 100 * np.mean(idf == idf.max())
    print(f"зерно {seed}: найнижчий IDF -> {lowest}")
    print(f"          найвищий IDF {idf.max():.2f}, і його має {at_top:.1f} % словника")

Найдешевші слова корпусу — «не», «для», «у», «з». Це очікувано: службові слова є
скрізь, і саме їх IDF мусить придушити.

А от **«вдалося»** серед найдешевших — не очікувано. Подивимось на нього окремо.

In [ ]:
frequency = collections.Counter()
for text in documents:
    frequency.update(tokenize(text.lower()))

print("найчастіші слова всього корпусу:")
for rank, (word, times) in enumerate(frequency.most_common(8), start=1):
    print(f"  {rank}. {word:<10} {times:>6} ужитків")

print()
print("усього слововживань:", sum(frequency.values()))
print("різних словоформ:   ", len(frequency))

«Вдалося» — на пʼятому місці за частотою в усьому корпусі, поряд із «не», «для»,
«у» і «з». У звичайній українській воно ніде близько до пʼятірки не стоїть.

Причина проста: наш корпус зібрано з перекладів інтерфейсів, а інтерфейси більшу
частину часу повідомляють про **невдачі**: «не вдалося відкрити», «не вдалося
зберегти», «не вдалося зʼєднатись». Тема 02 показала те саме з іншого боку: її
токенізатор розрізав «телефон» на чотири шматки, а «налаштування» лишив цілим.
Обидва заміри кажуть одне: **модель знає домен, а не мову**.

## 6 · Головне питання теми: чи змінює IDF відповідь

Питання формулюємо так. Візьмемо документ і спитаємо, яке слово в ньому головне.
Дві відповіді: за самою частотою (мішок слів із теми 04) і за TF-IDF. Чи це те саме
слово?

Тут є пастка, у яку легко впасти. Наші документи короткі, і в переважній більшості
**всі слова трапляються рівно по разу**. У такому документі «найчастіше слово» не
існує: частота не має думки взагалі, і `argmax` поверне просто перше слово за
абеткою. Порівнювати з ним безглуздо.

Тому рахуємо чесно й окремо:

* у скількох документах у частоти взагалі є думка (унікальний максимум);
* серед **тільки цих** документів — у скількох IDF цю думку перекриває.

In [ ]:
def main_word_disagreement(texts):
    vec, counts = count_matrix(texts)
    tfidf, weights = fit_tfidf(texts, vocabulary=vec.vocabulary_)
    silent = unique = overridden = naive_same = total = 0
    for row in range(counts.shape[0]):
        start, stop = counts.indptr[row], counts.indptr[row + 1]
        if start == stop:
            continue                                    # документ без жодного слова
        total += 1
        columns = counts.indices[start:stop]
        times = counts.data[start:stop]
        best_by_tfidf = columns[int(np.argmax(weights.data[start:stop]))]
        # «наївний» спосіб: беремо argmax частоти й не питаємо, чи була там нічия.
        # Саме так рахують зазвичай — і саме тому число виходить оманливим
        naive_same += int(columns[int(np.argmax(times))] == best_by_tfidf)
        leaders = columns[times == times.max()]          # усі слова з максимальною частотою
        if len(leaders) > 1:
            silent += 1                                  # частота не має думки
        else:
            unique += 1
            if best_by_tfidf != leaders[0]:
                overridden += 1
    return silent, unique, overridden, naive_same, total

for seed in SEEDS:
    silent, unique, overridden, naive_same, total = main_word_disagreement(sample_documents(seed))
    print(f"зерно {seed}:")
    print(f"   наївно: головне слово те саме у {100*naive_same/total:.1f} % документів, "
          f"тобто IDF нібито міняє відповідь у {100*(1-naive_same/total):.1f} %")
    print(f"   але частота мовчить (усі слова по разу) у {silent} із {total} "
          f"документів — це {100*silent/total:.1f} %")
    print(f"   там, де вона говорить ({unique} док.), IDF перекриває її "
          f"у {100*overridden/unique:.1f} % випадків")

Ось два числа теми, і вони кажуть різне про одне й те саме.

**Перше.** У 91 із кожних 100 документів частота слів не дає жодної підказки:
всі слова по разу. Там TF-IDF — це фактично **чиста IDF**, і саме вона обирає
головне слово одноосібно.

**Друге.** У решті документів, де частота таки має улюбленця, IDF відбирає в нього
перше місце більш ніж у чотирьох випадках із пʼяти.

Разом: IDF — не косметичний множник. На короткому тексті вона **і є** відповіддю.

## 7 · Своїми руками: жодної магії всередині

Найкорисніша перевірка практики — написати те саме самому й переконатись, що
бібліотека рахує рівно те, що написано у формулі.

Кроки такі:

1. `df(w)` — у скількох документах слово трапилось.
2. `idf(w) = ln((1 + N) / (1 + df(w))) + 1`.
3. Помножити кожну клітинку матриці частот на IDF її колонки.
4. Поділити кожен рядок на його довжину (L2-нормалізація).

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n_documents = counts.shape[0]

# крок 1: у скількох документах трапилось кожне слово
document_frequency = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)

# крок 2: обернена документна частота зі згладжуванням, як у sklearn
our_idf = np.log((1 + n_documents) / (1 + document_frequency)) + 1.0

# крок 3: множимо частоти на вагу слова, крок 4: рівняємо довжину рядків
our_tfidf = normalize(counts.multiply(our_idf).tocsr())

# те саме бібліотекою, з тим самим словником
library, library_tfidf = fit_tfidf(texts, vocabulary=vec.vocabulary_)

assert np.allclose(our_idf, library.idf_), "IDF розійшлася!"
assert np.allclose(our_tfidf.toarray()[:200], library_tfidf.toarray()[:200]), "матриця розійшлася!"
print("✅ збігається: максимальна різниця IDF", float(np.max(np.abs(our_idf - library.idf_))))

## 8 · Чому логарифм, а не просто N / df

Ідея IDF — «рідкісне цінніше за часте» — сама по собі логарифма не вимагає.
Найпростіше було б узяти `N / df`. Заміряємо, що з цього вийде.

Міряти будемо так: у кожному документі порахуємо, яку **частку всієї ваги
документа** забирає його найважче слово. Якщо схема справедлива, вага розподілена
між словами; якщо схема зривається — одне слово забирає майже все.

In [ ]:
def top_word_share(counts, idf):
    """Середня частка ваги, яку в документі забирає одне найважче слово."""
    weighted = counts.multiply(idf).tocsr()
    shares = []
    for row in range(weighted.shape[0]):
        start, stop = weighted.indptr[row], weighted.indptr[row + 1]
        if start == stop:
            continue
        row_weights = weighted.data[start:stop]
        shares.append(row_weights.max() / row_weights.sum())
    return float(np.mean(shares))

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    n = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    schemes = {
        "без IDF":        np.ones_like(df),
        "log(N/df) + 1":  np.log((1 + n) / (1 + df)) + 1.0,
        "корінь N/df":    np.sqrt(n / df),
        "лінійна N/df":   n / df,
    }
    print(f"зерно {seed}:")
    for name, idf in schemes.items():
        span = idf.max() / idf.min()
        print(f"   {name:<15} частка головного слова {top_word_share(counts, idf):.4f}"
              f"   розмах ваг {span:>8.1f}x")

Число, заради якого все це рахувалось: **без IDF** одне слово забирає 20 % ваги
документа, з логарифмом — 26 %, а з лінійною `N/df` — **61 %**.

Тобто лінійна версія перетворює документ на одне слово: найрідкісніше. Решта тексту
просто перестає впливати. І видно, звідки це береться: розмах ваг у логарифмічної
схеми — приблизно 4.5 рази між найдешевшим і найдорожчим словом, а в лінійної —
понад пʼять тисяч разів.

Логарифм тут не для краси. Він робить рівно одне: **перетворює множення на
додавання**. Слово, у десять разів рідкісніше, стає не в десять разів важливішим,
а на однакову добавку важливішим. Це і є те, чого ми хочемо від «цінності».

## 9 · Згладжування: одиниці зверху й знизу

У формулі `sklearn` є дві одиниці, і вони роблять різні речі.

`+1` у **чисельнику й знаменнику** (`smooth_idf=True`) — це «уявний документ, у
якому є всі слова». Він рятує від ділення на нуль, коли `df = 0`.

Ділення на нуль здається неможливим: як слово може бути у словнику й нізде не
траплятись? Легко — якщо словник узято з іншого корпусу. Заміряємо, наскільки це
часта ситуація.

`+1` **зовні логарифма** — інше: воно не дає вазі впасти в нуль для слова, яке є
геть у всіх документах. Без нього таке слово зникло б із матриці зовсім.

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)

smoothed = np.log((1 + n) / (1 + df)) + 1.0
plain    = np.log(n / df) + 1.0

print(f"максимальний IDF: зі згладжуванням {smoothed.max():.4f}, без {plain.max():.4f}")
print(f"мінімальний IDF:  зі згладжуванням {smoothed.min():.4f}, без {plain.min():.4f}")
print(f"середня різниця по словнику {np.mean(np.abs(smoothed - plain)):.4f}, "
      f"найбільша {np.max(np.abs(smoothed - plain)):.4f}")
print(f"слів, що трапились рівно в одному документі: {100*np.mean(df == 1):.1f} % словника")

# а тепер — словник із іншої вибірки того самого корпусу
other_vec, _ = count_matrix(sample_documents(1))
unseen = [w for w in other_vec.vocabulary_ if w not in vec.vocabulary_]
print()
print(f"слів чужого словника, яких у цій вибірці немає: {len(unseen)} із "
      f"{len(other_vec.vocabulary_)} ({100*len(unseen)/len(other_vec.vocabulary_):.1f} %)")
print("для них df = 0: без згладжування це ln(N/0) = нескінченність,")
print(f"зі згладжуванням — скінченні ln({n+1}/1) + 1 = {math.log((n+1)/1)+1:.4f}")

Майже тридцять відсотків. Тобто якщо ти навчив `TfidfVectorizer` на одній вибірці й
приніс словник на іншу, майже третина слів матиме `df = 0`. Без згладжування це
не «трохи неточно», а `inf` у матриці й `NaN` після нормалізації.

Друге, що робить згладжування, — трохи підрізає верх: 10.90 стає 10.21. На пошук
це, як ми зараз побачимо, майже не впливає, і чесно про це сказати важливіше, ніж
вигадати різницю.

## 10 · Нормалізація: чому довгий документ не має вигравати

Скалярний добуток запиту й документа росте разом із документом: що більше слів,
то більше доданків. Без нормалізації пошук перетворюється на «покажи найдовші
тексти, у яких трапилось потрібне слово».

L2-нормалізація ділить вектор документа на його довжину. Після цього довжина
кожного вектора — рівно одиниця, а скалярний добуток стає **косинусом кута**:
мірою напрямку, а не розміру.

Заміряємо на пошуку. Задача чесна й перевірювана: беремо документ, робимо з нього
запит із трьох слів і дивимось, на якому місці знайдеться сам документ.

In [ ]:
def build_queries(counts, seed, n_queries=300, mode="mixed"):
    """З кожного документа-мішені робимо запит із трьох його слів.
    mode='mixed'  — два найчастіші в корпусі слова документа плюс одне найрідкісніше
                    (так виглядає справжній запит: службові слова плюс змістовне);
    mode='random' — три випадкові слова документа."""
    rng = np.random.default_rng(1000 + seed)
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    lengths = np.diff(counts.indptr)
    targets = rng.choice(np.where(lengths >= 6)[0], n_queries, replace=False)
    rows, columns = [], []
    for number, target in enumerate(targets):
        start, stop = counts.indptr[target], counts.indptr[target + 1]
        terms = counts.indices[start:stop]
        if mode == "mixed":
            by_df = terms[np.argsort(-df[terms])]
            picked = [by_df[0], by_df[1], by_df[-1]]
        else:
            picked = rng.choice(terms, 3, replace=False)
        for term in picked:
            rows.append(number)
            columns.append(term)
    queries = sp.csr_matrix((np.ones(len(rows)), (rows, columns)),
                            shape=(n_queries, counts.shape[1]))
    return queries, targets

def search_quality(counts, queries, targets, idf, use_l2):
    documents_matrix = counts.multiply(idf).tocsr()
    queries_matrix = queries.multiply(idf).tocsr()
    if use_l2:
        documents_matrix = normalize(documents_matrix)
        queries_matrix = normalize(queries_matrix)
    scores = (queries_matrix @ documents_matrix.T).toarray()
    own = scores[np.arange(len(targets)), targets]
    # місце мішені: скільки документів набрало більше (нічиї ріжемо за номером)
    better = (scores > own[:, None]).sum(axis=1)
    ties_before = ((scores == own[:, None]) &
                   (np.arange(counts.shape[0])[None, :] < targets[:, None])).sum(axis=1)
    place = 1 + better + ties_before
    lengths = np.diff(counts.indptr)
    top_ten = np.argsort(-scores, axis=1)[:, :10]
    return dict(mrr=float(np.mean(1 / place)),
                first=float(np.mean(place == 1)),
                top_length=float(lengths[top_ten].mean()))

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    n = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    idf = np.log((1 + n) / (1 + df)) + 1.0
    queries, targets = build_queries(counts, seed)
    lengths = np.diff(counts.indptr)
    print(f"зерно {seed}: середня довжина документа {lengths.mean():.2f} слова")
    for use_l2 in (False, True):
        r = search_quality(counts, queries, targets, idf, use_l2)
        label = "з L2" if use_l2 else "без L2"
        print(f"   {label:<6} MRR {r['mrr']:.4f}  знайдено першим {r['first']:.4f}  "
              f"середня довжина топ-10: {r['top_length']:.2f} слова")

Середній документ корпусу — сім слів. Без нормалізації пошук піднімає нагору
документи по 44-59 слів, тобто **у сім-девʼять разів довші за середній**, і знаходить
те, що просили, першим приблизно в 38 % випадків замість 68 %. Це не тонке
налаштування, а умова, щоб пошук узагалі працював.

## 11 · Пошук: де різниця видима, а де ні

Тепер порівняємо самі схеми ваг на тій самій задачі. І — важливо — на **двох різних
типах запитів**, бо відповідь від них залежить.

In [ ]:
def compare_schemes(mode):
    collected = collections.defaultdict(list)
    for seed in SEEDS:
        texts = sample_documents(seed)
        vec, counts = count_matrix(texts)
        n = counts.shape[0]
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
        queries, targets = build_queries(counts, seed, mode=mode)
        schemes = {
            "без IDF":      np.ones_like(df),
            "TF-IDF (log)": np.log((1 + n) / (1 + df)) + 1.0,
            "без згладж.":  np.log(n / df) + 1.0,
            "лінійна N/df": n / df,
        }
        for name, idf in schemes.items():
            collected[name].append(search_quality(counts, queries, targets, idf, True))
    print(f"запити типу «{mode}»:")
    for name, runs in collected.items():
        mrr = [r["mrr"] for r in runs]
        print(f"   {name:<14} MRR {np.mean(mrr):.4f}  розкид по зернах {max(mrr)-min(mrr):.4f}"
              f"   знайдено першим {np.mean([r['first'] for r in runs]):.4f}")
    base = [r["mrr"] for r in collected["без IDF"]]
    best = [r["mrr"] for r in collected["TF-IDF (log)"]]
    gains = [b - a for b, a in zip(best, base)]
    print(f"   виграш TF-IDF над частотою по зернах: "
          + ", ".join(f"{g:+.4f}" for g in gains))
    return collected

mixed = compare_schemes("mixed")
print()
random_terms = compare_schemes("random")

Три висновки, і один із них незручний.

**Перший.** На запиті «два звичні слова плюс одне змістовне» — тобто на тому, як
люди справді пишуть запити, — TF-IDF виграє в частоти **0.22 MRR**, і виграш
приблизно однаковий на всіх трьох зернах. Він утричі більший за розкид самої
метрики, тобто справжній.

**Другий.** Лінійна `N/df` програє логарифмічній помітно й стабільно. Те саме, що
показала частка ваги: коли одне слово забирає все, пошук ламається.

**Третій, незручний.** На запиті з трьох випадкових слів документа виграш падає
до кількох сотих і **міняється у вісім разів між зернами**: на одному з трьох він
фактично нульовий. Це не «трохи менший виграш», а відсутність надійного висновку.
Причина видна з першого досліду: випадкові слова короткого документа й так переважно
рідкісні, і зважувати там нічого.

TF-IDF потрібна не «завжди», а тоді, коли **в запиті поруч стоять часте й рідкісне
слово**. Тоді вона й вирішує, яке з них слухати.

Подивимось на це очима — на конкретних запитах до тих самих 20 000 документів.

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
idf = np.log((1 + n) / (1 + df)) + 1.0
by_tfidf = normalize(counts.multiply(idf).tocsr())
by_count = normalize(counts.astype(float))
words = vec.get_feature_names_out()

def show(query, how_many=3):
    q = vec.transform([query])
    print("ЗАПИТ:", query)
    print("  ваги слів запиту:",
          ", ".join(f"{words[j]} idf {idf[j]:.2f}" for j in q.indices))
    for name, matrix, weighted in (("за частотою", by_count, False),
                                   ("за TF-IDF  ", by_tfidf, True)):
        qq = normalize(q.multiply(idf).tocsr()) if weighted else normalize(q.astype(float))
        scores = np.asarray((qq @ matrix.T).todense()).ravel()
        best = np.argsort(-scores)[:how_many]
        print(f"  {name}:")
        for j in best:
            print(f"     {scores[j]:.3f}  {' '.join(texts[j].split())[:58]}")
    print()

show("дані про принтер")
show("не вдалося прочитати каталог")

Перший запит — різниця видима одразу: за частотою нагору вилазить документ
«Дані:» (два слова, одне з них із запиту — і в короткому документі це дає високий
косинус), а TF-IDF ставить першим «Отримання інформації про принтер». Слово
«принтер» рідкісне, і воно тягне.

Другий запит — різниця не видима зовсім: обидві схеми дають ту саму трійку.
Коли в запиті всі слова однаково рідкісні, зважувати нічого.

## 12 · Де TF-IDF ламається · випадок 1: синоніми

Тут нам щастить із корпусом. Той самий англійський рядок різні перекладачі
переклали по-різному — і ми маємо пари документів, про які **точно** відомо, що
вони означають одне й те саме, бо походять з одного оригіналу.

Це ідеальна перевірка. Якщо TF-IDF розуміє зміст, косинус таких пар має бути високим.

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка,
    у яких майже немає спільних слів."""
    by_source = collections.defaultdict(dict)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source][' '.join(target.split())] = program
    found = []
    for source, variants in by_source.items():
        texts = list(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                first, second = texts[i], texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(tokenize(first.lower()))
                b = set(tokenize(second.lower()))
                if len(a) < 3 or len(b) < 3:
                    continue
                overlap = len(a & b) / len(a | b)
                if overlap < 0.34:
                    found.append((first, second, len(a & b)))
    return found

pairs = paraphrase_pairs()
print(f"пар «те саме іншими словами»: {len(pairs)}, "
      f"з них без жодного спільного слова: {sum(1 for p in pairs if p[2] == 0)}")
print()
for first, second, shared in pairs[:4]:
    print(f"  спільних слів {shared}")
    print(f"    A: {first}")
    print(f"    B: {second}")

In [ ]:
for seed in SEEDS:
    texts = sample_documents(seed)
    tfidf, _ = fit_tfidf(texts)
    left = tfidf.transform([p[0] for p in pairs])
    right = tfidf.transform([p[1] for p in pairs])
    cosine = np.asarray(left.multiply(right).sum(axis=1)).ravel()

    # контроль: випадкові пари документів корпусу, які нічого спільного не мають
    rng = np.random.default_rng(7 + seed)
    a = tfidf.transform([texts[i] for i in rng.choice(len(texts), len(pairs))])
    b = tfidf.transform([texts[i] for i in rng.choice(len(texts), len(pairs))])
    random_cosine = np.asarray(a.multiply(b).sum(axis=1)).ravel()

    # для порівняння: ті самі пари, але подані символьними триграмами
    char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 3))
    char_tfidf.fit(texts)
    ca = char_tfidf.transform([p[0] for p in pairs])
    cb = char_tfidf.transform([p[1] for p in pairs])
    char_cosine = np.asarray(ca.multiply(cb).sum(axis=1)).ravel()

    print(f"зерно {seed}: синоніми — середній косинус {cosine.mean():.4f}, "
          f"медіана {np.median(cosine):.4f}, рівно нуль у {100*np.mean(cosine == 0):.1f} %")
    print(f"          випадкові пари — {random_cosine.mean():.4f}; "
          f"символьні триграми на тих самих парах — {char_cosine.mean():.4f}")

Ось і поломка, і вона повна. Пари документів, які означають **буквально одне й те
саме**, дістають середній косинус близько 0.22, а більш ніж кожна пʼята з них —
**рівно нуль**, тобто рівно стільки, скільки два випадкові документи корпусу.

Причина не в налаштуваннях. TF-IDF порівнює **написання**, а не зміст: слова
«регулярний» і «формальний» для неї — дві незалежні колонки без жодного звʼязку.
Ніяким підбором ваг це не лікується, бо зміст у поданні просто відсутній.

Символьні триграми ловлять більше (0.48 проти 0.22) — але лише тому, що вони
бачать спільні корені й закінчення, а не тому, що розуміють синонімію. Справжня
відповідь на це питання — блок 3 курсу, де слово стане вектором
(ембединги, тема 12 · Word2Vec).

## 13 · Випадок 2: дуже короткі документи

Другий спосіб зламати TF-IDF — дати їй короткий текст. Заміряємо дві речі:
яку частку довжини вектора несе одне слово, і наскільки вектор змінюється, якщо
одне слово прибрати.

In [ ]:
BUCKETS = [(1, 3), (4, 6), (7, 10), (11, 20), (21, 10**6)]

for seed in SEEDS:
    texts = sample_documents(seed)
    vec, counts = count_matrix(texts)
    n = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    idf = np.log((1 + n) / (1 + df)) + 1.0
    unit = normalize(counts.multiply(idf).tocsr())     # рядки одиничної довжини
    lengths = np.asarray(counts.sum(axis=1)).ravel().astype(int)
    rng = np.random.default_rng(50 + seed)

    biggest, after_loss = [], []
    for row in range(unit.shape[0]):
        start, stop = unit.indptr[row], unit.indptr[row + 1]
        if start == stop:
            biggest.append(np.nan); after_loss.append(np.nan); continue
        row_weights = unit.data[start:stop]
        biggest.append(row_weights.max())               # вектор одиничний, тож це і є частка
        if stop - start == 1:
            after_loss.append(0.0); continue
        damaged = row_weights.copy()
        damaged[rng.integers(0, stop - start)] = 0      # прибираємо одне слово
        after_loss.append(float(row_weights @ damaged / np.linalg.norm(damaged)))
    biggest = np.array(biggest); after_loss = np.array(after_loss)

    print(f"зерно {seed}:")
    for low, high in BUCKETS:
        mask = (lengths >= low) & (lengths <= high) & np.isfinite(biggest)
        name = f"{low}-{high}" if high < 10**6 else "21+"
        print(f"   {name:<6} слів: {mask.sum():>5} док. ({100*mask.mean():>4.1f} %) | "
              f"головне слово несе {biggest[mask].mean():.4f} довжини вектора | "
              f"косинус із собою без одного слова {np.nanmean(after_loss[mask]):.4f}")

У документі на три слова одне слово несе **73 %** довжини вектора, і втрата одного
слова роняє схожість із самим собою до 0.74. У документі на 21 слово й більше —
39 % і 0.98.

Практичний наслідок: на коротких текстах TF-IDF **дуже чутлива до випадковості**.
Одна інша словоформа, один синонім, одна одруківка — і документ поїхав. А наш
корпус саме такий: половина документів має від чотирьох до шести слів.

## 14 · Випадок 3: рідкісне в корпусі — не те саме, що рідкісне в мові

IDF не знає мови. Вона знає рівно один корпус — той, на якому її порахували.
Слово, якого в цьому корпусі мало, дістане велику вагу, навіть якщо в житті
його знає кожна дитина.

Перевіримо просто: візьмемо звичайнісінькі українські слова й подивимось на їхню
`df` у нашому корпусі.

In [ ]:
texts = sample_documents(0)
vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
idf = np.log((1 + n) / (1 + df)) + 1.0

everyday = ['мама', 'хліб', 'дощ', 'сонце', 'дерево', 'кіт', 'любов', 'вулиця', 'пісня', 'брат']
technical = ['налаштування', 'файл', 'помилка', 'каталог', 'сервер']

print("побутові українські слова:")
for word in everyday:
    if word in vec.vocabulary_:
        j = vec.vocabulary_[word]
        print(f"   {word:<10} df={int(df[j]):<5} idf={idf[j]:.4f}")
    else:
        print(f"   {word:<10} у корпусі немає взагалі")
print()
print("технічні слова:")
for word in technical:
    j = vec.vocabulary_[word]
    print(f"   {word:<14} df={int(df[j]):<5} idf={idf[j]:.4f}")

Девʼять із десяти побутових слів у корпусі **не трапляються жодного разу**. Єдине,
що прорвалось, — «дерево», і то не рослина, а елемент інтерфейсу; воно дістає
IDF 8.0132, тобто важить **більше за «сервер»** з його 6.5468.

Тепер порахуємо це не на десяти словах, а систематично. Візьмемо одну програму з
власним доменом — векторний редактор Inkscape — і порівняємо IDF її слів,
порахований усередині цього домену, з IDF, порахованим на решті корпусу.

In [ ]:
inkscape = [target for program, _, target in corpus if program == 'inkscape']
outside_all = [target for program, _, target in corpus if program != 'inkscape']

if len(inkscape) < 200:
    print("у цьому корпусі немає Inkscape — розділ пропущено")
else:
    rng = np.random.default_rng(0)
    outside = [outside_all[i] for i in rng.choice(len(outside_all), N_DOCS, replace=False)]
    inside_vec, inside_counts = count_matrix(inkscape)
    outside_vec, outside_counts = count_matrix(outside)

    def idf_of(counts):
        n = counts.shape[0]
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
        return np.log((1 + n) / (1 + df)) + 1.0, df

    inside_idf, inside_df = idf_of(inside_counts)
    outside_idf, outside_df = idf_of(outside_counts)
    missing_weight = math.log((N_DOCS + 1) / 1) + 1      # слова, якого зовні немає взагалі

    uses = np.asarray(inside_counts.sum(axis=0)).ravel()
    words = inside_vec.get_feature_names_out()
    top = np.argsort(-uses)[:400]

    rows = []
    for j in top:
        word = words[j]
        if word in outside_vec.vocabulary_:
            outer = float(outside_idf[outside_vec.vocabulary_[word]])
        else:
            outer = missing_weight
        rows.append((word, int(uses[j]), float(inside_idf[j]), outer, outer - float(inside_idf[j])))

    gaps = [r[4] for r in rows]
    print(f"документів Inkscape: {len(inkscape)}")
    print(f"серед 400 найуживаніших слів Inkscape середній розрив IDF "
          f"«зовні мінус усередині»: {np.mean(gaps):.4f}")
    print()
    rows.sort(key=lambda r: -r[4])
    for word, times, inner, outer, gap in rows[:8]:
        print(f"   {word:<14} ужитків у Inkscape {times:>4} | IDF свій {inner:.4f} | "
              f"IDF чужий {outer:.4f} | розрив {gap:+.4f}")

Слово «контур» — звичайне українське слово й буденний термін у векторному
редакторі — дістає від зовнішнього корпусу **максимальну можливу вагу**, бо там
воно не трапляється взагалі. Середній розрив по чотирьохстах найуживаніших
словах домену — близько 1.8 одиниці IDF.

Це і є причина, чому IDF **не можна переносити між доменами**, і чому «стоп-слова»
корисно рахувати самим, а не брати готовий список: у нашому корпусі «вдалося» —
службове слово, а в корпусі новин воно нормальне.

## 15 · Підсумок блоку 1 в числах

Пʼять тем — одна лінія: **текст не таблиця → ріжемо на шматки → зводимо форми →
рахуємо → зважуємо**. Порахуємо тут головні числа кожного кроку, щоб побачити їх
поруч.

In [ ]:
t0 = time.time()

# ── тема 01: закони природної мови ──────────────────────────────────
ranks = np.arange(1, len(frequency) + 1)
freqs = np.array(sorted(frequency.values(), reverse=True), dtype=float)
window = ranks <= 10000
zipf_slope = np.polyfit(np.log(ranks[window]), np.log(freqs[window]), 1)[0]
# нахил дуже чутливий до вікна підгонки: на рангах 1-1000 виходить -0.85,
# на всьому хвості -1.39. Тому вікно називаємо явно.
hapax = 100 * np.mean(freqs == 1)

seen, total_tokens, xs, ys = set(), 0, [], []
for index, text in enumerate(documents):
    for word in tokenize(text.lower()):
        seen.add(word)
        total_tokens += 1
    if (index + 1) % 2000 == 0:
        xs.append(total_tokens); ys.append(len(seen))
heaps_beta = np.polyfit(np.log(xs), np.log(ys), 1)[0]

print(f"01 · нахил Ципфа на рангах 1-10000: {zipf_slope:.4f} (природна мова близько -1)")
print(f"01 · слів, що трапились рівно раз: {hapax:.1f} %")
print(f"01 · показник Гіпса beta = {heaps_beta:.4f} (словник росте й не насичується)")

# ── тема 02: та сама думка двома мовами ─────────────────────────────
ukrainian_chars = sum(len(target) for _, _, target in corpus)
english_chars = sum(len(source) for _, source, _ in corpus)
if english_chars > 0:
    print(f"02 · той самий зміст українською довший у символах у "
          f"{ukrainian_chars / english_chars:.4f} раза")

# ── тема 03: скільки форм в однієї леми ─────────────────────────────
cyrillic = list(frequency)     # шаблон і так лишає самі українські слова
try:
    import pymorphy3
    analyzer = pymorphy3.MorphAnalyzer(lang='uk')
    lemmas = set(analyzer.parse(word)[0].normal_form for word in cyrillic)
    print(f"03 · словоформ {len(cyrillic)}, лем {len(lemmas)}, "
          f"падіння словника {100 * (1 - len(lemmas) / len(cyrillic)):.1f} %")
except Exception as error:
    print("03 · pymorphy3 недоступний:", error)

# ── тема 04: розрідженість ──────────────────────────────────────────
vec, counts = count_matrix(sample_documents(0))
cells = counts.shape[0] * counts.shape[1]
dense_gb = cells * 8 / 1024**3
sparse_mb = (counts.data.nbytes + counts.indices.nbytes + counts.indptr.nbytes) / 1024**2
print(f"04 · матриця {counts.shape[0]} x {counts.shape[1]}, заповнено "
      f"{100*counts.nnz/cells:.4f} %")
print(f"04 · щільно це {dense_gb:.2f} ГБ, розріджено {sparse_mb:.2f} МБ "
      f"({dense_gb*1024/sparse_mb:.0f} разів різниці)")
print("     (у самої теми 04 на її вибірці виходить 968 разів — матриця там менша)")

# ── тема 05: що додала вага ─────────────────────────────────────────
silent, unique, overridden, naive_same, total = main_word_disagreement(sample_documents(0))
print(f"05 · частота мовчить у {100*silent/total:.1f} % документів; "
      f"де говорить — IDF перекриває {100*overridden/unique:.1f} %")
print()
print(f"(розділ рахувався {time.time() - t0:.1f} с)")

Прочитати цю таблицю варто так.

**01.** Мова підпорядкована степеневим законам: нахил Ципфа близько −1, третина
слів трапляється рівно раз, словник росте як корінь із тексту й не насичується.
Саме тому «просто перелічити всі слова» не працює ніколи.

**02.** Той самий зміст українською довший, і токенізатор ріже його на більше
шматків. Але ріже він за доменом, а не за мовою — і TF-IDF успадкувала цю
властивість цілком: «вдалося» дешеве саме тому, що корпус про помилки.

**03.** Зведення форм до леми ріже словник більш ніж удвічі. Для TF-IDF це прямо
корисно: без лематизації «файл», «файла» й «файлу» — три незалежні колонки, і
кожна отримує свою IDF, розмазуючи вагу того самого поняття на три частини.

**04.** Матриця розріджена до чотирьох сотих відсотка. TF-IDF нічого тут не міняє:
вона переписує значення в тих самих клітинках.

**05.** І нарешті вага. Вона робить велику роботу — міняє головне слово в більшості
документів, дає 0.19 MRR на реалістичних запитах — і не робить ніякої на трьох
випадкових словах. І вона не бачить синонімів узагалі.

Це і є межа блоку 1. Усе, що ми будували пʼять тем, тримається на **збігу
написань**. Наступний блок навчиться на цьому поданні розвʼязувати справжні задачі —
класифікацію, пошук, тематичне моделювання, — а третій нарешті замінить збіг
написань на схожість змісту.

## 16 · Дані, на яких працюють фігури лекції

Фігури в лекції рахують TF-IDF просто в браузері — на 120 справжніх документах
нашого корпусу й на справжніх `df`, узятих із тих самих 20 000. Ця клітинка
друкує числа, які там показано, щоб їх можна було звірити.

In [ ]:
allowed = re.compile(r"^[А-Яа-яІіЇїЄєҐґʼ ,.:\-]+$")
texts = sample_documents(0)
unique_texts = {}
for text in texts:
    tidy = ' '.join(text.split())
    if 4 <= len(tokenize(tidy.lower())) <= 8 and len(tidy) <= 58 \
       and allowed.fullmatch(tidy) and tidy.lower() not in unique_texts:
        unique_texts[tidy.lower()] = tidy
selected = list(unique_texts.values())

# шість документів, у яких слово повторюється, а IDF усе одно міняє головне слово:
# без них у фігурі 1 нічого було б показати, бо в коротких текстах нічия майже завжди
FOR_FIGURE_ONE = [
    "Не можна вилучати пакунки з групи, якої не існує",
    "Вибрати пакунки за назвою, а не за здатністю.",
    "Визначити час витримки для калібрування для зеленого",
    "Адреса не є абсолютною, і не вказано базової адреси",
    "Атрибут членства у групах у мережі",
    "Довільні дані програми, якими підписано дані",
]
mini = selected[::max(1, len(selected) // 120)][:120]
mini = FOR_FIGURE_ONE + [t for t in mini if t not in FOR_FIGURE_ONE]

vec, counts = count_matrix(texts)
n = counts.shape[0]
df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)

def figure_idf(d, kind):
    if kind == 'none':
        return 1.0
    if kind == 'log':
        return math.log((1 + n) / (1 + d)) + 1
    if kind == 'root':
        return math.sqrt(n / d)
    return n / d

def mini_share(kind):
    shares = []
    for text in mini:
        seen = collections.Counter(w for w in tokenize(text.lower()) if w in vec.vocabulary_)
        weights = [times * figure_idf(df[vec.vocabulary_[word]], kind)
                   for word, times in seen.items()]
        shares.append(max(weights) / sum(weights))
    return float(np.mean(shares))

print("мінікорпус фігур:", len(mini), "документів")
for kind, name in (('none', 'без IDF'), ('log', 'log(N/df)+1'),
                   ('root', 'корінь N/df'), ('lin', 'лінійна N/df')):
    span = figure_idf(1, kind) / figure_idf(n, kind)
    print(f"   {name:<13} частка головного слова {mini_share(kind):.4f}"
          f"   розмах ваг від df=1 до df=N: {span:>8.1f}x")

silent = agree = overridden = 0
for text in mini:
    seen = collections.Counter(w for w in tokenize(text.lower()) if w in vec.vocabulary_)
    leaders = [w for w in seen if seen[w] == max(seen.values())]
    best = max(seen, key=lambda w: (seen[w] * figure_idf(df[vec.vocabulary_[w]], 'log'), w))
    if len(leaders) > 1:
        silent += 1
    elif leaders[0] == best:
        agree += 1
    else:
        overridden += 1
print(f"   частота мовчить у {silent}, збігається у {agree}, "
      f"перекрита IDF у {overridden} документах")
print()
print("перші шість документів мінікорпусу — саме ті, де IDF перекриває частоту:")
for text in mini[:6]:
    seen = collections.Counter(w for w in tokenize(text.lower()) if w in vec.vocabulary_)
    leaders = [w for w in seen if seen[w] == max(seen.values())]
    best = max(seen, key=lambda w: (seen[w] * figure_idf(df[vec.vocabulary_[w]], 'log'), w))
    print(f"   {text}")
    print(f"      частота: {leaders[0]} ({max(seen.values())} рази, df "
          f"{int(df[vec.vocabulary_[leaders[0]]])}) -> TF-IDF: {best} "
          f"(df {int(df[vec.vocabulary_[best]])})")

## Завдання

Повний текст із критеріями «зроблено» — у [homework.md](homework.md). Коротко:

### 🟢 Рівень 1

Порахуй IDF на **своєму** тексті: 200-1000 коротких документів (заголовки, назви
товарів, теми листів — що завгодно). Надрукуй десять слів із найменшою IDF і десять
із найбільшою.

**Зроблено, якщо** ти назвав хоча б одне слово з нижньої десятки, яке потрапило туди
через **домен**, а не через мову — як «вдалося» в нашому корпусі.

### 🟡 Рівень 2

Повтори замір із розділу 8 для сімейства `idf = (N/df) ** p`, де `p` пробігає
0, 0.1, …, 1.0. Побудуй графік «`p` → частка ваги головного слова» на трьох зернах
і постав на нього лінію логарифмічної схеми.

**Зроблено, якщо** ти назвав, при якому приблизно `p` крива перетинає цю лінію, і
пояснив, чому логарифм поводиться як маленьке `p`, а не як `p = 1`.

### 🔴 Рівень 3

Реалізуй **BM25** з нуля й порівняй його з TF-IDF на задачі з розділів 10-11:
три зерна, обидва типи запитів, метрика MRR, `k1` і `b` — хоча б по три значення.

**Зроблено, якщо** ти назвав виграш або програш **разом із розкидом по зернах** і
сказав прямо, чи більший він за розкид. Відповідь «різниці немає» є повноцінною
відповіддю, якщо вона підкріплена числами.